In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

In [ ]:
# Task 1: Write your code here:
# Loading the data into the dataframe df_delivery
delivery_path = os.path.join(path, 'Q1_data.csv')
df_delivery = pd.read_csv(delivery_path)
df_delivery

In [ ]:
# Task 2: Write your code here:
df_delivery.head() # Showing the first 5 samples

In [ ]:
# Task 3: Write your code here:
df_delivery.info()
# I noticed there are some null rows in some cols and some categorial cols

In [ ]:
# Task 4: Write your code here:
df_delivery.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df_delivery['Delivery_Time'].dropna(), bins=70, edgecolor='black')
plt.title('delivery time Distribution')
plt.xlabel('delivery time')
plt.ylabel('Frequency')
plt.show()
# Notice how the graph is a bit skewed to the right and it has a mean of approx. 56

In [ ]:
# Task 1: Write your code here:
df_delivery = df_delivery.drop(columns=['Order_ID'])
df_delivery.columns

In [ ]:
# Task 2: Write your code here:
# let's take a look by using isnull.sum
print(df_delivery.isnull().sum())
# weather trafficlevel timeofday courier and delivery time has some missing data
df_delivery.info()
# first let's fix the weather, traffic level and time of day col by filling the null rows with the most frequent in data (mode)

df_delivery['Weather'] = df_delivery['Weather'].fillna(df_delivery['Weather'].mode()[0])
df_delivery['Traffic_Level'] = df_delivery['Traffic_Level'].fillna(df_delivery['Traffic_Level'].mode()[0])
df_delivery['Time_of_Day'] = df_delivery['Time_of_Day'].fillna(df_delivery['Time_of_Day'].mode()[0])

# fix the courier exp years by filling the mode also since the data has integer number of years
df_delivery['Courier_Experience_yrs'] = df_delivery['Courier_Experience_yrs'].fillna(df_delivery['Courier_Experience_yrs'].mode()[0])

# finally with the deliv time we must drop them since we cannot predict without them
df_delivery = df_delivery.dropna(subset=['Delivery_Time'])


print(df_delivery.isnull().sum())


In [ ]:
# Task 3: Write your code here:
duplicate_rows = df_delivery.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows. Removing them...")
    df_delivery.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")

In [ ]:
# Task 4: Write your code here:

categorical_cols = df_delivery.select_dtypes(include=["object"]).columns # Check and select our categorical cols
print(categorical_cols)

le = LabelEncoder()
print("Applying Label encoder Encoding")
for i in categorical_cols:
  df_delivery[i] = le.fit_transform(df_delivery[i])

df_delivery.head()

In [ ]:
# Task 5: Write your code here:
# We shouldn't do this now to avoid data leakage but since u asked...
all_cols = df_delivery.columns.drop("Delivery_Time")

# scale the features
scaler = StandardScaler()
df_delivery[all_cols] = scaler.fit_transform(df_delivery[all_cols])
df_delivery.head()


In [ ]:
# Task 6: Write your code here:
# Target imbalance ? this is a linear regression problem and the target doesn't classes here

In [ ]:
# Task 1: Write your code here:
X = df_delivery[all_cols]
y = df_delivery['Delivery_Time']
X.head()

In [ ]:
y.head()

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error

model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)


kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
rmse_scores = []
ypred = []
ytest = []
for fold, (train_idx, test_idx) in enumerate(kfold.split(X), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    # print shapes
    model.fit(X_train, y_train)
    y_fold_pred = model.predict(X_test)
    ytest.append(y_test)
    ypred.append(y_fold_pred)
    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_test, y_fold_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_test, y_fold_pred)))
    print("-" * 30)


mae_scores = np.array(mae_scores)
rmse_scores = np.array(rmse_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")
print(f"RMSE: ${rmse_scores.mean():,.2f}")

In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(6, 4))
plt.scatter(ytest, ypred, alpha=0.7)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], 'r--', linewidth=2)
plt.xlabel("Actual y_test (Ground Truth)")
plt.ylabel("Predicted y_pred (Linear Regression)")
plt.title("Linear Regression: Predictions vs. Ground Truth")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here: